In [ ]:
import os
import numpy as np
import pandas as pd
import glob
from tqdm.notebook import tqdm

In [ ]:
def fast_process_edge_feat(edges_df, num_nodes, num_timesteps, node_col, feat_col, max_edges):
    # 1. Get rank of each edge (0, 1, 2...) per timestep/node
    # This replaces the pad_list logic
    edge_ranks = edges_df.groupby(['timestep', node_col]).cumcount()
    
    # 2. Only keep edges within our max limit
    mask = edge_ranks < max_edges
    filtered_edges = edges_df[mask]
    ranks = edge_ranks[mask].values
    
    # 3. Extract indices for fast assignment
    # We assume node_idx and timestep are 0-indexed. 
    # If not, map them: nodes_df['node_idx'].map(mapping_dict)
    t_idx = filtered_edges['timestep'].values
    n_idx = filtered_edges[node_col].values
    vals = filtered_edges[feat_col].values
    
    # 4. Pre-allocate and fill
    out = np.zeros((num_timesteps, num_nodes, max_edges), dtype=np.float32)
    out[t_idx, n_idx, ranks] = vals
    return out

def load_data(events, edge_index_2d, edge_index_1d, model="model_2"):
    # Pre-calculate max edges once
    max_e_2d_from = edge_index_2d.groupby("from_node").size().max()
    max_e_2d_to = edge_index_2d.groupby("to_node").size().max()
    max_e_1d_from = edge_index_1d.groupby("from_node").size().max()
    max_e_1d_to = edge_index_1d.groupby("to_node").size().max()

    if not os.path.exists(model): os.makedirs(model)
    data = []
    for event in tqdm(events):
        sample = {}
        
        # --- 2D DOMAIN ---
        nodes_2d = pd.read_csv(f"{event}/2d_nodes_dynamic_all.csv")
        nodes_2d = nodes_2d.sort_values(['timestep', 'node_idx']).reset_index(drop=True)
        edges_2d = pd.read_csv(f"{event}/2d_edges_dynamic_all.csv").merge(edge_index_2d, on="edge_idx")
        is_sorted = ((edges_2d['timestep'].diff().fillna(0) >= 0) | 
                     ((edges_2d['timestep'].diff().fillna(0) == 0) & 
                      (edges_2d['from_node'].diff().fillna(0) >= 0))).all()
        
        if not is_sorted:
            print("WARNING: Data not sorted! Adding sort...")
        edges_2d = edges_2d.sort_values(['timestep', 'from_node']).reset_index(drop=True)
        num_n2d = nodes_2d.node_idx.max() + 1
        num_t = nodes_2d.timestep.max() + 1

        # Vectorized Node features
        sample["rainfall"] = nodes_2d.pivot(index='timestep', columns='node_idx', values='rainfall').values
        sample["water_level"] = nodes_2d.pivot(index='timestep', columns='node_idx', values='water_level').values
        sample["water_volume"] = nodes_2d.pivot(index='timestep', columns='node_idx', values='water_volume').values

        # Vectorized Edge features (The real speedup)
        for dirc, col, mx in [("from", "from_node", max_e_2d_from), ("to", "to_node", max_e_2d_to)]:
            for feat in ["flow", "velocity"]:
                key = f"{feat}_{dirc}_node_2d"
                sample[key] = fast_process_edge_feat(edges_2d, num_n2d, num_t, col, feat, mx)

        # --- 1D DOMAIN ---
        nodes_1d = pd.read_csv(f"{event}/1d_nodes_dynamic_all.csv")
        edges_1d = pd.read_csv(f"{event}/1d_edges_dynamic_all.csv").merge(edge_index_1d, on="edge_idx")
        
        num_n1d = nodes_1d.node_idx.max() + 1

        sample["1d_water_level"] = nodes_1d.pivot(index='timestep', columns='node_idx', values='water_level').values
        sample["inlet_flow"] = nodes_1d.pivot(index='timestep', columns='node_idx', values='inlet_flow').values
        
        for dirc, col, mx in [("from", "from_node", max_e_1d_from), ("to", "to_node", max_e_1d_to)]:
            for feat in ["flow", "velocity"]:
                key = f"{feat}_{dirc}_node_1d"
                sample[key] = fast_process_edge_feat(edges_1d, num_n1d, num_t, col, feat, mx)

        # Save logic
        event_id = event.split('/')[-1].split('_')[-1]
        event_path = os.path.join(model, f"event_{event_id}")
        if not os.path.exists(event_path): os.makedirs(event_path)
        
        for k, v in sample.items():
            np.save(f"{event_path}/{k}.npy", v.astype(np.float32))
 

In [ ]:
events = glob.glob(f"/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/event_*")

edges_1d_static = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/1d_edges_static.csv")
nodes_1d_static = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/1d_nodes_static.csv")
edges_2d_static = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/2d_edges_static.csv")
nodes_2d_static = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/2d_nodes_static.csv")
connection = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/1d2d_connections.csv")

edge_index_2d = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/2d_edge_index.csv")
edge_index_1d = pd.read_csv("/kaggle/input/datasets/jobayerhossain/flood-data/Dataset_Rerelease/Models/Model_2/train/1d_edge_index.csv")
if not os.path.exists('model_2'):
    os.mkdir("model_2")
edge_index_2d.to_csv("model_2/edge_index_2d.csv", index=False)
edge_index_1d.to_csv("model_2/edge_index_1d.csv", index=False)
edges_1d_static.to_csv("model_2/edges_1d_static.csv", index=False)
nodes_1d_static.to_csv("model_2/nodes_1d_static.csv", index=False)
edges_2d_static.to_csv("model_2/edges_2d_static.csv", index=False)
nodes_2d_static.to_csv("model_2/nodes_2d_static.csv", index=False)
connection.to_csv("model_2/1d2d_connection.csv", index=False)

data = load_data(events, edge_index_2d, edge_index_1d)